# Notebook 1 — FBref 데이터 수집

EPL, La Liga, Bundesliga, Serie A 선수 통계 (2017-18 ~ 2023-24) 를 FBref에서 수집합니다.

**출력:** `data/season_all.csv`

## 패키지 설치

In [ ]:
!pip install soccerdata -q
!pip install pandas numpy -q

## 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import soccerdata as sd
import os
import warnings
warnings.filterwarnings('ignore')

# 데이터 저장 경로
DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

## FBref 리더 설정

4개 리그, 7시즌 대상

In [ ]:
LEAGUES = [
    'ENG-Premier League',
    'ESP-La Liga',
    'GER-Bundesliga',
    'ITA-Serie A',
]
# soccerdata는 시즌을 시작 연도로 지정: 2017 = 2017-18 시즌
SEASONS = list(range(2017, 2024))  # 2017-18 ~ 2023-24

fbref = sd.FBref(leagues=LEAGUES, seasons=SEASONS)
print('FBref 리더 준비 완료')

## 각 스탯 카테고리 수집

스탯 유형별로 순차 수집 후 병합합니다. FBref 요청 제한으로 인해 시간이 걸릴 수 있어요.

In [ ]:
def flatten_df(df):
    """MultiIndex 컬럼/인덱스를 flat하게 변환"""
    df = df.reset_index()
    # MultiIndex 컬럼 처리
    if isinstance(df.columns, pd.MultiIndex):
        # 상위 레벨이 빈 문자열이면 하위 레벨만 사용
        df.columns = [
            col[1] if col[0] in ('', 'Unnamed: 0_level_0') else '_'.join(str(c) for c in col if c)
            for col in df.columns
        ]
    return df

In [ ]:
print('1/6 standard stats 수집 중...')
df_std = flatten_df(fbref.read_player_season_stats(stat_type='standard'))
print(f'  → {len(df_std):,}행, {len(df_std.columns)}컬럼')
print(df_std.columns.tolist()[:15])

In [ ]:
print('2/6 shooting stats 수집 중...')
df_sh = flatten_df(fbref.read_player_season_stats(stat_type='shooting'))
print(f'  → {len(df_sh):,}행, {len(df_sh.columns)}컬럼')

In [ ]:
print('3/6 passing stats 수집 중...')
df_pass = flatten_df(fbref.read_player_season_stats(stat_type='passing'))
print(f'  → {len(df_pass):,}행, {len(df_pass.columns)}컬럼')

In [ ]:
print('4/6 defense stats 수집 중...')
df_def = flatten_df(fbref.read_player_season_stats(stat_type='defense'))
print(f'  → {len(df_def):,}행, {len(df_def.columns)}컬럼')

In [ ]:
print('5/6 possession stats 수집 중...')
df_poss = flatten_df(fbref.read_player_season_stats(stat_type='possession'))
print(f'  → {len(df_poss):,}행, {len(df_poss.columns)}컬럼')

In [ ]:
print('6/6 misc + keepers stats 수집 중...')
df_misc = flatten_df(fbref.read_player_season_stats(stat_type='misc'))
df_gk   = flatten_df(fbref.read_player_season_stats(stat_type='keepers'))
print(f'  misc → {len(df_misc):,}행  |  keepers → {len(df_gk):,}행')

## 데이터 병합

모든 카테고리를 (league, season, team, player) 기준으로 outer merge 합니다.

In [ ]:
# 병합 키 탐지 (soccerdata 버전에 따라 컬럼명이 다를 수 있음)
def detect_merge_keys(df):
    candidates = ['league', 'season', 'team', 'player']
    found = [c for c in candidates if c in [col.lower() for col in df.columns]]
    # 실제 컬럼명(대소문자 그대로) 반환
    col_map = {col.lower(): col for col in df.columns}
    return [col_map[c] for c in found]

merge_keys = detect_merge_keys(df_std)
print('병합 키:', merge_keys)

In [ ]:
def safe_merge(left, right, keys):
    """중복 컬럼을 제거하며 merge"""
    right_keys_set = set(keys)
    right_cols = keys + [c for c in right.columns if c not in left.columns and c not in right_keys_set]
    return pd.merge(left, right[right_cols], on=keys, how='outer')

dfs = [df_std, df_sh, df_pass, df_def, df_poss, df_misc]
season_all = dfs[0].copy()
for df in dfs[1:]:
    season_all = safe_merge(season_all, df, merge_keys)

# GK 스탯 추가 (GK 선수만 값 존재, 나머지는 NaN → 이후 0으로)
gk_keys = detect_merge_keys(df_gk)
gk_extra = [c for c in df_gk.columns if c not in season_all.columns and c not in set(gk_keys)]
if gk_extra:
    season_all = pd.merge(season_all, df_gk[gk_keys + gk_extra], on=gk_keys, how='left')

print(f'병합 완료: {season_all.shape}')

## 컬럼 정리 및 이름 표준화

In [ ]:
# 현재 컬럼명 확인
print(season_all.columns.tolist())

In [ ]:
# 컬럼명 소문자 통일 후 핵심 메타 컬럼 확인
col_lower = {c: c.lower() for c in season_all.columns}

# league, season, team, player 컬럼을 League/Season/Team/Player로 통일
rename_map = {}
for col in season_all.columns:
    if col.lower() == 'league':  rename_map[col] = 'League'
    elif col.lower() == 'season': rename_map[col] = 'Season'
    elif col.lower() == 'team':   rename_map[col] = 'Team'
    elif col.lower() == 'player': rename_map[col] = 'Player'

season_all = season_all.rename(columns=rename_map)
print('메타 컬럼:', [c for c in ['League','Season','Team','Player'] if c in season_all.columns])

## 클리닝: 출전 시간 필터 + NaN 처리

In [ ]:
# 출전 시간(분) 컬럼 찾기
min_col = None
for c in season_all.columns:
    if c.lower() in ('min', 'minutes', 'playing_time_min', 'playing time_min'):
        min_col = c
        break

if min_col is None:
    # 수동 확인용
    print('출전 시간 컬럼 후보:')
    print([c for c in season_all.columns if 'min' in c.lower()])
else:
    print(f'출전 시간 컬럼: {min_col}')
    # 출전 시간 숫자 변환
    season_all[min_col] = pd.to_numeric(season_all[min_col], errors='coerce')
    # 270분(약 3경기) 미만 제거
    before = len(season_all)
    season_all = season_all[season_all[min_col] >= 270].copy()
    print(f'{before:,} → {len(season_all):,}행 (270분 미만 제거)')

In [ ]:
# 숫자 컬럼 NaN → 0
num_cols = season_all.select_dtypes(include='number').columns
season_all[num_cols] = season_all[num_cols].fillna(0)

print(f'최종: {season_all.shape}')
print(f'NaN 잔여: {season_all.isnull().sum().sum()}')
season_all.head(3)

## 저장

In [ ]:
out_path = os.path.join(DATA_DIR, 'season_all.csv')
season_all.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장 완료: {out_path}')
print(f'  행: {len(season_all):,}  |  컬럼: {len(season_all.columns)}')